## Goal
Audit one scene through the Phase 2 production APIs. This notebook does not parse COLMAP or implement camera math.

## Setup
Set `BTS_SCENE_ROOT` to select another scene. The default is the local HCM0181 public scene.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np

repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))
scene_root = Path(os.environ.get('BTS_SCENE_ROOT', repo_root / 'data/phase1/public_set/HCM0181'))
scene_root

WindowsPath('C:/Users/USER/Desktop/digital-twin-3D/data/phase1/public_set/HCM0181')

## Steps
Build the serving manifest and the separate diagnostic record.

In [2]:
from bts_nvs.data import SceneDataset, build_scene_diagnostics, build_scene_manifest

manifest = build_scene_manifest(scene_root)
diagnostics = build_scene_diagnostics(scene_root)
dataset = SceneDataset(manifest, scene_root)

## Checks
Summaries cover camera/output dimensions, distortion, sparse support, reprojection, tracks, and a bounded image sample.

In [3]:
sample_count = min(8, len(dataset))
sample_means = [dataset[index].image.mean(axis=(0, 1)) for index in range(sample_count)]
camera_centers = manifest.train_camera_to_world[:, :3, 3]
radial_coefficients = [item.coefficients[0] for item in manifest.train_distortion if item.model == 'SIMPLE_RADIAL']
summary = {
    'scene_id': manifest.scene_id,
    'train_images': len(manifest.train_image_names),
    'test_cameras': len(manifest.test_image_names),
    'test_resolutions': sorted({(item.width, item.height) for item in manifest.test_intrinsics}),
    'distortion_models': sorted({item.model for item in manifest.train_distortion}),
    'k1_range': (min(radial_coefficients), max(radial_coefficients)) if radial_coefficients else None,
    'camera_center_min': camera_centers.min(axis=0).tolist(),
    'camera_center_max': camera_centers.max(axis=0).tolist(),
    'sparse_points': len(manifest.sparse_points),
    'reprojection_error_p50_p90': np.percentile(diagnostics.reprojection_errors, [50, 90]).tolist(),
    'track_length_p50_p90': np.percentile(diagnostics.track_lengths, [50, 90]).tolist(),
    'train_support_p50_p90': np.percentile(diagnostics.train_support_counts, [50, 90]).tolist(),
    'sample_rgb_mean': np.mean(sample_means, axis=0).tolist(),
}
summary

{'scene_id': 'HCM0181',
 'train_images': 240,
 'test_cameras': 60,
 'test_resolutions': [(1320, 989)],
 'distortion_models': ['SIMPLE_RADIAL'],
 'k1_range': (0.00900353942750311, 0.00900353942750311),
 'camera_center_min': [-6.994681772613513,
  -7.857151411894659,
  -8.16721911164048],
 'camera_center_max': [7.430251465883325,
  4.0553055492759835,
  5.0767476718550775],
 'sparse_points': 188665,
 'reprojection_error_p50_p90': [1.173088196689492, 1.9137499931796704],
 'track_length_p50_p90': [4.0, 13.0],
 'train_support_p50_p90': [3.0, 9.0],
 'sample_rgb_mean': [124.46734467398964,
  125.93390745166528,
  123.32145360327236]}

## Next Steps
Investigate only values that violate the manifest contract or differ materially across scenes; keep exploratory statistics out of `SceneManifest`.